# Publish "Just Use Postgres" Blog to GitHub Pages

This notebook publishes the finalized Markdown blog post at `content/just-use-postgres.md` into the
existing GitHub Pages site served from the `docs/` folder. It:

1. Converts the Markdown to a styled HTML subpage under `docs/blog/` using the site's `style.css`.
2. Copies the four referenced visual assets from `content/visuals/` into `docs/blog/visuals/` and
   rewrites the image `src` paths (preserving alt text and captions).
3. Links the new page from the main index (`docs/index.html`) in the same format as existing posts.
4. Preserves frontmatter metadata (title, author, date) as HTML `<title>` / meta tags.
5. Makes **local file changes only** — no `git add`, `commit`, or `push`.

## 1. Import Required Libraries

Standard-library modules for filesystem paths (`pathlib`), text processing (`re`), and binary-safe
asset copying (`shutil`). Markdown-to-HTML conversion uses the `markdown` package; if it is not
installed, the cell installs it via `pip`.

In [1]:
import re
import shutil
import sys
import subprocess
from datetime import date
from pathlib import Path

try:
    import markdown
except ImportError:
    subprocess.run([sys.executable, "-m", "pip", "install", "markdown"], check=True)
    import markdown

print("markdown version:", markdown.__version__)

markdown version: 3.10.2


## 2. Define Workspace Paths and Constants

Resolve the source blog file, the content visuals folder, the `docs/` site root, the `docs/blog/`
output folder, and `docs/style.css`. The published slug is derived from the source filename
(`just-use-postgres`). The four visual filenames that must be copied and path-rewritten are listed
explicitly.

In [2]:
WORKSPACE = Path("/Users/shaileshmishra/my-docs/my-proj/how2genmodel")

SRC_BLOG = WORKSPACE / "content" / "just-use-postgres.md"
CONTENT_VISUALS = WORKSPACE / "content" / "visuals"
DOCS_ROOT = WORKSPACE / "docs"
BLOG_DIR = DOCS_ROOT / "blog"
BLOG_VISUALS = BLOG_DIR / "visuals"
STYLE_CSS = DOCS_ROOT / "style.css"

SLUG = SRC_BLOG.stem  # "just-use-postgres"
OUT_HTML = BLOG_DIR / f"{SLUG}.html"

VISUAL_FILES = [
    "db1-five-to-one-consolidation-hero.svg",
    "db1-one-engine-one-story-ops.png",
    "db1-four-breakpoints-panel.png",
    "db1-decision-rule-strip.svg",
]

for p in [SRC_BLOG, CONTENT_VISUALS, DOCS_ROOT, BLOG_DIR, STYLE_CSS]:
    print(f"{'OK     ' if p.exists() else 'MISSING'}  {p}")
print("Slug        :", SLUG)
print("Output HTML :", OUT_HTML)

OK       /Users/shaileshmishra/my-docs/my-proj/how2genmodel/content/just-use-postgres.md
OK       /Users/shaileshmishra/my-docs/my-proj/how2genmodel/content/visuals
OK       /Users/shaileshmishra/my-docs/my-proj/how2genmodel/docs
OK       /Users/shaileshmishra/my-docs/my-proj/how2genmodel/docs/blog
OK       /Users/shaileshmishra/my-docs/my-proj/how2genmodel/docs/style.css
Slug        : just-use-postgres
Output HTML : /Users/shaileshmishra/my-docs/my-proj/how2genmodel/docs/blog/just-use-postgres.html


## 3. Read and Parse the Blog Markdown Frontmatter

Read the source Markdown, split the YAML frontmatter from the body, and parse the simple
`key: value` pairs into a dictionary. Extract `title`, `author`, `description`, and `ms.date` for
use in the HTML `<title>` and meta tags. Also derive a human-readable date and an estimated read
time (word count / 225 wpm).

In [3]:
raw = SRC_BLOG.read_text(encoding="utf-8")

fm_match = re.match(r"^---\n(.*?)\n---\n(.*)$", raw, re.DOTALL)
assert fm_match, "Frontmatter block not found"
frontmatter_text, body_md = fm_match.group(1), fm_match.group(2)

meta = {}
for line in frontmatter_text.splitlines():
    if ":" in line:
        key, _, val = line.partition(":")
        meta[key.strip()] = val.strip().strip("'\"")

TITLE = meta.get("title", SLUG)
AUTHOR = meta.get("author", "Shailesh Mishra")
DESCRIPTION = meta.get("description", "")
DATE_ISO = meta.get("ms.date", "")

y, m, d = map(int, DATE_ISO.split("-"))
HUMAN_DATE = date(y, m, d).strftime("%B %-d, %Y")

word_count = len(re.findall(r"\w+", body_md))
READ_MIN = max(1, round(word_count / 225))

TAGS = ["postgresql", "databases", "architecture", "consolidation", "case-study"]

print("Title      :", TITLE)
print("Author     :", AUTHOR)
print("Date (ISO) :", DATE_ISO, "->", HUMAN_DATE)
print("Words      :", word_count, "->", READ_MIN, "min read")
print("Description:", DESCRIPTION[:90], "...")

Title      : Just Use Postgres — Until You Hit One of These Four Walls
Author     : Shailesh Mishra
Date (ISO) : 2026-06-26 -> June 26, 2026
Words      : 2706 -> 12 min read
Description: One ACID engine now absorbs vectors, time-series, queues, search, and documents. Here is t ...


## 4. Inspect the Existing Site Template and Style

Read `docs/index.html` and an existing post under `docs/blog/` to confirm the head section, the
`../style.css` stylesheet link convention, the `.site-header` / `.post-header` / `.post-content`
markup, and the `<ul class="blog-list">` card format. Reusing this exact structure keeps the new
page visually consistent with the rest of the site.

In [4]:
index_html = (DOCS_ROOT / "index.html").read_text(encoding="utf-8")
sample_page = (BLOG_DIR / "postgresql-explain-buffers-case-study.html").read_text(encoding="utf-8")

print("--- docs/index.html : <head> + first blog card ---")
print(index_html[:1900])
print("\n--- existing post : <head> + post-header ---")
print(sample_page[:1500])

--- docs/index.html : <head> + first blog card ---
<!DOCTYPE html>
<html lang="en">
<head>
  <meta charset="UTF-8">
  <meta name="viewport" content="width=device-width, initial-scale=1.0">
  <title>Shailesh Mishra - Blog</title>
  <meta name="description" content="Technical blog by Shailesh Mishra. AI code assistant optimization, context engineering, PostgreSQL performance, cloud engineering, and real-world case studies.">
  <link rel="stylesheet" href="style.css">
</head>
<body>

<header class="site-header">
  <a href="./" class="logo">Shailesh Mishra</a>
  <nav>
    <a href="./">Blog</a>
    <a href="https://github.com/sendtoshailesh">GitHub</a>
  </nav>
</header>

<main class="container">
  <div class="hero">
    <h1>Blog</h1>
    <p>Technical deep-dives, case studies, and field notes from working with customers on cloud and database engineering.</p>
  </div>

  <ul class="blog-list">
    <li class="blog-card">
      <h2><a href="blog/loop-engineering-ai-native-development.html">Fro

## 5. Determine the GitHub Pages Base URL

Derive the published base URL from the repo configuration. The git remote points at
`github.com/sendtoshailesh/content-creation`, and `docs/blog/README.md` documents the live site as
`https://sendtoshailesh.github.io/content-creation/`. If a `CNAME` file exists it takes precedence
(custom domain). The canonical page URL is `<base>/blog/<slug>.html`.

In [5]:
# Default base from repo name (project pages site) + README confirmation
PAGES_BASE = "https://sendtoshailesh.github.io/content-creation/"

readme = BLOG_DIR / "README.md"
if readme.exists():
    m = re.search(r"https://[\w.-]+\.github\.io/[\w./-]*", readme.read_text(encoding="utf-8"))
    if m:
        print("README base hint:", m.group(0))

# A CNAME (custom domain) would override the project-pages base
cname = DOCS_ROOT / "CNAME"
if cname.exists():
    PAGES_BASE = "https://" + cname.read_text(encoding="utf-8").strip().strip("/") + "/"
    print("CNAME override   :", PAGES_BASE)
else:
    print("No CNAME file; using project-pages base.")

CANONICAL_URL = PAGES_BASE.rstrip("/") + f"/blog/{SLUG}.html"
print("Pages base       :", PAGES_BASE)
print("Canonical URL    :", CANONICAL_URL)

README base hint: https://sendtoshailesh.github.io/content-creation/
No CNAME file; using project-pages base.
Pages base       : https://sendtoshailesh.github.io/content-creation/
Canonical URL    : https://sendtoshailesh.github.io/content-creation/blog/just-use-postgres.html


## 6. Convert Blog Markdown Body to HTML

Strip the leading `# Title` H1 from the body (the site template renders the title separately in the
`.post-header`), then convert the remaining Markdown to an HTML fragment with the `tables` and
`fenced_code` extensions enabled. Image captions written as `*italic*` lines become
`<p><em>…</em></p>`, matching the caption convention used by existing posts.

In [6]:
# Remove the first H1 (title is rendered by the template's .post-header)
body_no_h1 = re.sub(r"^#\s+.*\n", "", body_md, count=1).lstrip("\n")

md = markdown.Markdown(extensions=["tables", "fenced_code", "sane_lists"])
body_html = md.convert(body_no_h1)

print("Fragment length:", len(body_html), "chars")
print(body_html[:1400])

Fragment length: 19173 chars
<h1>Just Use Postgres — Until You Hit One of These Four Walls</h1>
<p>Across years of working with customers, I've lost count of how many teams I've helped untangle the
same knot. One sticks with me: a team running five moving parts to serve one product — Postgres for
the relational core, a standalone Redis for the hot path, Elasticsearch for search, a separate
message queue for background jobs, and a cron service to fire the scheduled work. Five datastores.
Five backup stories. Five things to monitor, patch, secure, and get paged about at 2 a.m.</p>
<p>We collapsed all of it into one Postgres.</p>
<p>Elasticsearch split into Postgres full-text search (<code>tsvector</code>) for lexical queries and
<a href="https://github.com/pgvector/pgvector">pgvector</a> for semantic search. The message queue moved into
a Postgres-native queue. The cron service became <code>pg_cron</code>. Redis's hot path folded back into the
database it was caching in the first place. 

## 7. Copy Visual Assets to the Site

Create `docs/blog/visuals/` if needed, then use `shutil.copy2` (binary-safe, preserves metadata) to
copy each of the four assets from `content/visuals/` into the site. The two SVGs and two PNGs are
copied byte-for-byte so the live site renders them correctly.

In [7]:
BLOG_VISUALS.mkdir(parents=True, exist_ok=True)

copied_assets = []
for name in VISUAL_FILES:
    src = CONTENT_VISUALS / name
    dst = BLOG_VISUALS / name
    assert src.exists(), f"Missing source asset: {src}"
    shutil.copy2(src, dst)
    copied_assets.append(dst)
    print(f"copied  {src}\n     -> {dst}  ({dst.stat().st_size} bytes)\n")

print("Copied", len(copied_assets), "assets into", BLOG_VISUALS)

copied  /Users/shaileshmishra/my-docs/my-proj/how2genmodel/content/visuals/db1-five-to-one-consolidation-hero.svg
     -> /Users/shaileshmishra/my-docs/my-proj/how2genmodel/docs/blog/visuals/db1-five-to-one-consolidation-hero.svg  (10448 bytes)

copied  /Users/shaileshmishra/my-docs/my-proj/how2genmodel/content/visuals/db1-one-engine-one-story-ops.png
     -> /Users/shaileshmishra/my-docs/my-proj/how2genmodel/docs/blog/visuals/db1-one-engine-one-story-ops.png  (728766 bytes)

copied  /Users/shaileshmishra/my-docs/my-proj/how2genmodel/content/visuals/db1-four-breakpoints-panel.png
     -> /Users/shaileshmishra/my-docs/my-proj/how2genmodel/docs/blog/visuals/db1-four-breakpoints-panel.png  (665933 bytes)

copied  /Users/shaileshmishra/my-docs/my-proj/how2genmodel/content/visuals/db1-decision-rule-strip.svg
     -> /Users/shaileshmishra/my-docs/my-proj/how2genmodel/docs/blog/visuals/db1-decision-rule-strip.svg  (5173 bytes)

Copied 4 assets into /Users/shaileshmishra/my-docs/my-proj/how2ge

## 8. Rewrite Image Source Paths in the HTML

The Markdown already references images as `visuals/<file>`. This step normalizes every `<img src>`
to `visuals/<basename>` (the relative path that resolves against the published page at
`docs/blog/<slug>.html` → `docs/blog/visuals/`), preserving the `alt` text and the surrounding
caption paragraphs. Before/after image tags are printed for verification.

In [8]:
print("BEFORE:")
for m in re.finditer(r"<img\b[^>]*>", body_html):
    print("  ", m.group(0))

# Rewrite every img src to visuals/<basename>, leaving alt and other attrs intact
body_html = re.sub(
    r'(<img\b[^>]*?\bsrc=")([^"]+)(")',
    lambda m: m.group(1) + "visuals/" + m.group(2).split("/")[-1] + m.group(3),
    body_html,
)

print("\nAFTER:")
for m in re.finditer(r"<img\b[^>]*>", body_html):
    print("  ", m.group(0))

BEFORE:
   <img alt="Five labeled workload boxes — vectors, queues, time-series, search, documents — converging into a single Postgres engine, each tagged with the extension that absorbs it: pgvector, pgmq, TimescaleDB, full-text, JSONB." src="visuals/db1-five-to-one-consolidation-hero.svg" />
   <img alt="Five duplicated operational stacks — backup, HA/failover, security model, on-call, skills to hire — collapsing from five copies down to one of each." src="visuals/db1-one-engine-one-story-ops.png" />
   <img alt="Four red-walled cards, each showing one ceiling number and the specialist it justifies: 7.5M inserts/sec at 4 ms P99 (ScyllaDB), all of YouTube for 5+ years (Vitess), single-digit-ms at 500k+ req/s and 99.999% (DynamoDB), ~1 billion rows/sec (ClickHouse)." src="visuals/db1-four-breakpoints-panel.png" />
   <img alt="Decision strip: default to Postgres, then ask whether you are hitting one of the four walls with a number attached — if yes, specialize and name the wall; if no,

## 9. Build the Full HTML Page from the Site Template

Assemble the complete document using the exact structure of existing posts: the `<head>` with
`<title>`, description/author/date meta, Open Graph tags, a canonical link, and the `../style.css`
stylesheet; the `.site-header` nav; the `.post-header` (title, byline, tags); and the converted
fragment inside `.post-content`.

In [9]:
import html as _html

tag_html = "\n        ".join(f'<span class="tag">{t}</span>' for t in TAGS)
desc_attr = _html.escape(DESCRIPTION, quote=True)
title_attr = _html.escape(TITLE, quote=True)

# Indent the converted fragment to sit cleanly inside .post-content
body_indented = "\n".join(("      " + ln) if ln.strip() else ln for ln in body_html.splitlines())

page = f"""<!DOCTYPE html>
<html lang="en">
<head>
  <meta charset="UTF-8">
  <meta name="viewport" content="width=device-width, initial-scale=1.0">
  <title>{title_attr} | {AUTHOR}</title>
  <meta name="description" content="{desc_attr}">
  <meta name="author" content="{AUTHOR}">
  <meta name="date" content="{DATE_ISO}">
  <meta property="og:title" content="{title_attr}">
  <meta property="og:description" content="{desc_attr}">
  <meta property="og:type" content="article">
  <meta property="og:url" content="{CANONICAL_URL}">
  <link rel="canonical" href="{CANONICAL_URL}">
  <link rel="stylesheet" href="../style.css">
</head>
<body>

<header class="site-header">
  <a href="../" class="logo">Shailesh Mishra</a>
  <nav>
    <a href="../">Blog</a>
    <a href="https://github.com/sendtoshailesh">GitHub</a>
  </nav>
</header>

<main class="container">
  <a href="../" class="back-link">&larr; Back to all posts</a>

  <article>
    <div class="post-header">
      <h1>{TITLE}</h1>
      <div class="meta">{AUTHOR} &middot; {HUMAN_DATE} &middot; {READ_MIN} min read</div>
      <div style="margin-top: 0.75rem;">
        {tag_html}
      </div>
    </div>

    <div class="post-content">

{body_indented}

    </div>
  </article>
</main>

<footer class="site-footer">
  &copy; 2026 Shailesh Mishra. Built with GitHub Pages.
</footer>

</body>
</html>
"""

print(page[:1500])
print("...\n[total page length:", len(page), "chars]")

<!DOCTYPE html>
<html lang="en">
<head>
  <meta charset="UTF-8">
  <meta name="viewport" content="width=device-width, initial-scale=1.0">
  <title>Just Use Postgres — Until You Hit One of These Four Walls | Shailesh Mishra</title>
  <meta name="description" content="One ACID engine now absorbs vectors, time-series, queues, search, and documents. Here is the consolidation case for defaulting to Postgres — and the four specific walls where I still reach for a specialist.">
  <meta name="author" content="Shailesh Mishra">
  <meta name="date" content="2026-06-26">
  <meta property="og:title" content="Just Use Postgres — Until You Hit One of These Four Walls">
  <meta property="og:description" content="One ACID engine now absorbs vectors, time-series, queues, search, and documents. Here is the consolidation case for defaulting to Postgres — and the four specific walls where I still reach for a specialist.">
  <meta property="og:type" content="article">
  <meta property="og:url" content="htt

## 10. Write the New Blog HTML Subpage

Write the assembled document to `docs/blog/<slug>.html` (`just-use-postgres.html`) and print the
created file path and size.

In [10]:
OUT_HTML.write_text(page, encoding="utf-8")
print("Wrote:", OUT_HTML)
print("Size :", OUT_HTML.stat().st_size, "bytes")

Wrote: /Users/shaileshmishra/my-docs/my-proj/how2genmodel/docs/blog/just-use-postgres.html
Size : 23222 bytes


## 11. Add the Blog Link to the Main Index Page

Insert a new `<li class="blog-card">` at the top of `<ul class="blog-list">` in `docs/index.html`
(newest first), matching the exact markup of existing entries — title link, meta line with date and
read time, excerpt, and tag spans. The operation is idempotent: it skips insertion if the slug is
already linked.

In [11]:
index_path = DOCS_ROOT / "index.html"
index_src = index_path.read_text(encoding="utf-8")

INDEX_EXCERPT = (
    "One ACID engine now absorbs vectors, time-series, queues, search, and documents. The "
    "consolidation case for defaulting to Postgres &mdash; and the four specific walls where a "
    "specialist still wins, each named with the number that justifies it."
)

new_card = (
    '    <li class="blog-card">\n'
    f'      <h2><a href="blog/{SLUG}.html">{TITLE}</a></h2>\n'
    f'      <div class="meta">{HUMAN_DATE} &middot; {READ_MIN} min read</div>\n'
    f'      <p class="excerpt">{INDEX_EXCERPT}</p>\n'
    '      <div class="tags">\n'
    + "\n".join(f'        <span class="tag">{t}</span>' for t in TAGS) + "\n"
    '      </div>\n'
    '    </li>\n'
)

anchor = '  <ul class="blog-list">\n'
assert anchor in index_src, "Could not find '<ul class=\"blog-list\">' anchor in index.html"

if f'blog/{SLUG}.html' in index_src:
    print("Index already links the post; no change made.")
else:
    index_src = index_src.replace(anchor, anchor + new_card, 1)
    index_path.write_text(index_src, encoding="utf-8")
    print("Inserted new blog card at top of <ul class=\"blog-list\"> in", index_path)

print("\n--- block added to docs/index.html ---")
print(new_card)

Inserted new blog card at top of <ul class="blog-list"> in /Users/shaileshmishra/my-docs/my-proj/how2genmodel/docs/index.html

--- block added to docs/index.html ---
    <li class="blog-card">
      <h2><a href="blog/just-use-postgres.html">Just Use Postgres — Until You Hit One of These Four Walls</a></h2>
      <div class="meta">June 26, 2026 &middot; 12 min read</div>
      <p class="excerpt">One ACID engine now absorbs vectors, time-series, queues, search, and documents. The consolidation case for defaulting to Postgres &mdash; and the four specific walls where a specialist still wins, each named with the number that justifies it.</p>
      <div class="tags">
        <span class="tag">postgresql</span>
        <span class="tag">databases</span>
        <span class="tag">architecture</span>
        <span class="tag">consolidation</span>
        <span class="tag">case-study</span>
      </div>
    </li>



## 12. Verify the Published Output and Report Results

Confirm the new HTML page and the four copied assets exist on disk, then print the final report:
the canonical URL, the new HTML file path, the four destination asset paths, and the index link
added. No `git add` / `commit` / `push` is performed — all changes are local.

In [12]:
def rel(p: Path) -> str:
    return str(p.relative_to(WORKSPACE))

print("=" * 70)
print("PUBLISH REPORT  —  Just Use Postgres")
print("=" * 70)

print("\nCanonical URL:")
print("  " + CANONICAL_URL)

print("\nNew HTML file:")
print(f"  {rel(OUT_HTML)}   ({'exists' if OUT_HTML.exists() else 'MISSING'})")

print("\nCopied asset paths (docs/blog/visuals/):")
for name in VISUAL_FILES:
    dst = BLOG_VISUALS / name
    print(f"  {rel(dst)}   ({'exists' if dst.exists() else 'MISSING'})")

print("\nIndex link added to docs/index.html (top of <ul class=\"blog-list\">):")
print(f'  <li class="blog-card"> ... <a href="blog/{SLUG}.html">{TITLE}</a> ... </li>')

print("\nGit: no add / commit / push performed — local file changes only.")
print("=" * 70)

PUBLISH REPORT  —  Just Use Postgres

Canonical URL:
  https://sendtoshailesh.github.io/content-creation/blog/just-use-postgres.html

New HTML file:
  docs/blog/just-use-postgres.html   (exists)

Copied asset paths (docs/blog/visuals/):
  docs/blog/visuals/db1-five-to-one-consolidation-hero.svg   (exists)
  docs/blog/visuals/db1-one-engine-one-story-ops.png   (exists)
  docs/blog/visuals/db1-four-breakpoints-panel.png   (exists)
  docs/blog/visuals/db1-decision-rule-strip.svg   (exists)

Index link added to docs/index.html (top of <ul class="blog-list">):
  <li class="blog-card"> ... <a href="blog/just-use-postgres.html">Just Use Postgres — Until You Hit One of These Four Walls</a> ... </li>

Git: no add / commit / push performed — local file changes only.
